In [149]:
import json
import os

In [150]:
!ls possible_answer

BFCL_v3_java.json
BFCL_v3_javascript.json
BFCL_v3_live_multiple.json
BFCL_v3_live_parallel.json
BFCL_v3_live_parallel_multiple.json
BFCL_v3_live_simple.json
BFCL_v3_multi_turn_base.json
BFCL_v3_multi_turn_composite_unused.json
BFCL_v3_multi_turn_long_context.json
BFCL_v3_multi_turn_miss_func.json
BFCL_v3_multi_turn_miss_param.json
BFCL_v3_multiple.json
BFCL_v3_parallel.json
BFCL_v3_parallel_multiple.json
BFCL_v3_simple.json
BFCL_v3_sql_unused.json


In [151]:
path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"

In [152]:
question_answer_pairs = {}
smallset = ["BFCL_v3_live_simple.json", "BFCL_v3_java.json", "BFCL_v3_javascript.json", "BFCL_v3_simple.json"]
# Traverse the directory and match files to the pair format
for file in os.listdir(path):
    if file in smallset and file.endswith(".json"):
        question_path = os.path.join(path, file)
        answer_path = os.path.join(path, "possible_answer", file)
        output_path = os.path.join(path, "paired_xlam_formatted", f"xlam_{file.split('.')[0]}.jsonl")
        
        question_answer_pairs[question_path] = {
            "question_path": question_path,
            "answer_path": answer_path,
            "output_path": output_path
        }


In [153]:
import json
from tqdm import tqdm

# === Helper Functions ===
def flatten_parameters(func):
    """Flatten the nested parameters structure."""
    return func.get("parameters", {}).get("properties", {})

def parse_ground_truth(gt_entry):
    """Extract tool name and arguments from the ground_truth entry."""
    try:
        name, args = list(gt_entry.items())[0]
        flat_args = {k: v[0] if isinstance(v, list) and len(v) > 0 else v for k, v in args.items()}
        return name, flat_args
    except (IndexError, KeyError, AttributeError) as e:
        # Return None to indicate a parsing error
        return None, None

def format_tool(func):
    """Format the function into a tool dictionary for XLAM."""
    return {
        "name": func["name"],
        "description": func.get("description", ""),
        "parameters": flatten_parameters(func)
    }

# === Main Processing ===
def main():
    # Iterate over each question-answer pair
    for paths in question_answer_pairs.values():
        QUESTION_PATH = paths["question_path"]
        ANSWER_PATH = paths["answer_path"]
        OUTPUT_PATH = paths["output_path"]

        # Load questions and answers from JSONL files (line by line)
        questions = []
        answers = []
        with open(QUESTION_PATH, "r") as fq:
            for line in fq:
                line = line.strip()
                if line:
                    questions.append(json.loads(line))
        with open(ANSWER_PATH, "r") as fa:
            for line in fa:
                line = line.strip()
                if line:
                    answers.append(json.loads(line))

        # Ensure equal number of question and answer entries
        assert len(questions) == len(answers), "Mismatched number of question and answer entries!"

        # Collect all unique tools from the dataset (deduplication by tool name)
        tool_pool = {}
        for q in questions:
            for f in q["function"]:
                tool_pool[f["name"]] = f

        all_tools = [format_tool(f) for f in tool_pool.values()]

        # Convert each question and corresponding answer into XLAM format
        failed_count = 0
        with open(OUTPUT_PATH, "w") as fout:
            for entry, gt in tqdm(zip(questions, answers), total=len(questions)):
                try:
                    query = entry["question"][0][0]["content"]
                    tool_call_name, tool_args = parse_ground_truth(gt["ground_truth"][0])
                    
                    # Skip entries with parsing errors
                    if tool_call_name is None:
                        failed_count += 1
                        continue
                    
                    xlam_entry = {
                        "id": entry["id"],
                        "query": query,
                        "answers": json.dumps([{"name": tool_call_name, "arguments": tool_args}]),
                        "tools": json.dumps(all_tools)
                    }
                    fout.write(json.dumps(xlam_entry) + "\n")
                except Exception as e:
                    # Print error info and continue with next entry
                    print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
                    failed_count += 1
                    continue

        print(f"✅ Done writing XLAM file for {QUESTION_PATH}. Failed entries: {failed_count}/{len(questions)}")

if __name__ == "__main__":
    main()

100%|██████████| 50/50 [00:00<00:00, 4142.52it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_javascript.json. Failed entries: 0/50


100%|██████████| 400/400 [00:00<00:00, 671.31it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_simple.json. Failed entries: 0/400


100%|██████████| 100/100 [00:00<00:00, 3117.31it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_java.json. Failed entries: 0/100


100%|██████████| 258/258 [00:00<00:00, 2507.58it/s]

✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_simple.json. Failed entries: 0/258


In [154]:
for path in question_answer_pairs.keys():
    answer = question_answer_pairs[path]["output_path"]
    with open(answer, "r") as f:
        print(len(f.readlines()))


50
400
100
258


# Convert to Tianhang Format

In [155]:
import os
question_answer_pairs_2 = {}
path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"
smallset = ["BFCL_v3_live_simple.json", "BFCL_v3_java.json", "BFCL_v3_javascript.json", "BFCL_v3_simple.json", "BFCL_v3_live_multiple.json"]
# Traverse the directory and match files to the pair format
for file in os.listdir(path):
    if file in smallset and file.endswith(".json"):
        question_path = os.path.join(path, file)
        answer_path = os.path.join(path, "possible_answer", file)
        output_path = os.path.join(path, "paired_tianhang_formatted", f"{file.split('.')[0]}.jsonl")
        
        question_answer_pairs_2[question_path] = {
            "question_path": question_path,
            "answer_path": answer_path,
            "output_path": output_path
        }


In [156]:
SYSTEM_PROMPT = """
You are an expert in composing functions. You are given a question and a set of possible functions.
Based on the question, you will need to make one or more function/tool calls to achieve the purpose.
If none of the function can be used, point it out. If the given question lacks the parameters required by the function, also point it out. You should only return the function call in tools call sections.
"""

FORMAT_VALUE = "Please return the function call(s) in JSON format. If you decide to return the function call(s), NO other text MUST be included. If you decide not to return any function call, return an empty list or a string saying 'No tools are suitable for this request.'"

USER_MESSAGE_FOR_CHAT_MODEL = "Questions:{user_prompt}\nHere is a list of functions in JSON format that you can invoke:\n{functions}. Should you decide to return the function call(s), NO other text MUST be included."


In [157]:
import json
from tqdm import tqdm

# === Helper Functions ===
def flatten_parameters(func):
    """Flatten the nested parameters structure."""
    return func.get("parameters", {}).get("properties", {})

def parse_ground_truth(gt_entry):
    """Extract tool name and arguments from the ground_truth entry."""
    try:
        name, args = list(gt_entry.items())[0]
        flat_args = {k: v[0] if isinstance(v, list) and len(v) > 0 else v for k, v in args.items()}
        return name, flat_args
    except (IndexError, KeyError, AttributeError) as e:
        # Return None to indicate a parsing error
        return None, None

def format_tool(func):
    """Format the function into a tool dictionary for XLAM."""
    return {
        "name": func["name"],
        "description": func.get("description", ""),
        "parameters": flatten_parameters(func)
    }


# === Main Processing ===
def main():
    # Iterate over each question-answer pair
    for paths in question_answer_pairs_2.values():
        QUESTION_PATH = paths["question_path"]
        ANSWER_PATH = paths["answer_path"]
        OUTPUT_PATH = paths["output_path"]

        # Load questions and answers from JSONL files (line by line)
        questions = []
        answers = []
        with open(QUESTION_PATH, "r") as fq:
            for line in fq:
                line = line.strip()
                if line:
                    questions.append(json.loads(line))
        with open(ANSWER_PATH, "r") as fa:
            for line in fa:
                line = line.strip()
                if line:
                    answers.append(json.loads(line))

        # Ensure equal number of question and answer entries
        assert len(questions) == len(answers), "Mismatched number of question and answer entries!"

        # Collect all unique tools from the dataset (deduplication by tool name)
        tool_pool = {}
        for q in questions:
            for f in q["function"]:
                tool_pool[f["name"]] = f

        all_tools = [format_tool(f) for f in tool_pool.values()]

        # Convert each question and corresponding answer into the specified format
        failed_count = 0
        with open(OUTPUT_PATH, "w") as fout:
            for entry, gt in tqdm(zip(questions, answers), total=len(questions)):
                try:
                    query = entry["question"][0][0]["content"]
                    tool_call_name, tool_args = parse_ground_truth(gt["ground_truth"][0])
                    
                    # Skip entries with parsing errors
                    if tool_call_name is None:
                        failed_count += 1
                        continue
                    
                    # Create the new format dictionary
                    new_format_entry = {
                        "format": FORMAT_VALUE,
                        "system_prompt": SYSTEM_PROMPT,
                        "user_query": [query],
                        "plan": "",
                        "tool_defs": all_tools,
                        "skill_trajectory": [{"name": tool_call_name, "arguments": tool_args}],
                        "reason_for_skills": [],
                        "expected_result": [],
                        "execution":[]
                    }
                    
                    fout.write(json.dumps(new_format_entry) + "\n")
                except Exception as e:
                    # Print error info and continue with next entry
                    print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
                    failed_count += 1
                    continue

        print(f"✅ Done writing XLAM file for {QUESTION_PATH}. Failed entries: {failed_count}/{len(questions)}")

if __name__ == "__main__":
    main()

100%|██████████| 50/50 [00:00<00:00, 1935.05it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_javascript.json. Failed entries: 0/50


100%|██████████| 1053/1053 [00:01<00:00, 733.34it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_multiple.json. Failed entries: 0/1053


100%|██████████| 400/400 [00:00<00:00, 1036.87it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_simple.json. Failed entries: 0/400


100%|██████████| 100/100 [00:00<00:00, 4794.81it/s]


✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_java.json. Failed entries: 0/100


100%|██████████| 258/258 [00:00<00:00, 4087.14it/s]

✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_simple.json. Failed entries: 0/258


In [158]:
file_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/paired_tianhang_formatted/BFCL_v3_live_multiple.jsonl"
output_file_path = "ex.json"

try:
    with open(file_path, "r") as file:
        content = file.read()
        num_lines = content.count('\n')
        print(f"The number of lines in the file is: {num_lines}")
        
        # Write the first line to ex.json
        first_line = content.split('\n', 1)[0]
        with open(output_file_path, "w") as output_file:
            output_file.write(first_line + '\n')
except FileNotFoundError:
    print(f"File not found: {file_path}")
except Exception as e:
    print(f"An error occurred while reading the file: {str(e)}")


The number of lines in the file is: 1053


In [159]:
import json

first_line_json = json.loads(first_line)
tool_defs = first_line_json.get("tool_defs")

In [160]:
len(tool_defs)

457

# Smart Augmentation

In [161]:
import os
question_answer_pairs_3 = {}
path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"
smallset = ["BFCL_v3_live_multiple.json"]
# Traverse the directory and match files to the pair format
for file in os.listdir(path):
    if file in smallset and file.endswith(".json"):
        question_path = os.path.join(path, file)
        answer_path = os.path.join(path, "possible_answer", file)
        output_path = os.path.join(path, "paired_tianhang_formatted", f"{file.split('.')[0]}_selective_augmentation.jsonl")
        
        question_answer_pairs_3[question_path] = {
            "question_path": question_path,
            "answer_path": answer_path,
            "output_path": output_path
        }


In [162]:
import json
import random
from tqdm import tqdm

# === Helper Functions ===
def flatten_parameters(func):
    """Flatten the nested parameters structure."""
    return func.get("parameters", {}).get("properties", {})

def format_tool(func):
    """Format the function into a tool dictionary for XLAM."""
    return {
        "name": func["name"],
        "description": func.get("description", ""),
        "parameters": flatten_parameters(func)
    }

# === Main Processing ===
def main():
    path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"
    for path in question_answer_pairs_3.values():
        QUESTION_PATH = path["question_path"]
        OUTPUT_PATH = path["output_path"]
        
        print(f"Processing {QUESTION_PATH}")
        
        # First, collect all unique tools from the dataset to create the tool pool
        tool_pool = {}
        questions = []
        with open(QUESTION_PATH, "r") as fq:
            for line in fq:
                line = line.strip()
                if line:
                    entry = json.loads(line)
                    questions.append(entry)
                    # Add tools to the pool
                    for f in entry["function"]:
                        if f["name"] not in tool_pool:
                            tool_pool[f["name"]] = f

        # Convert tool pool to list of formatted tools
        pool_tools = [format_tool(f) for f in tool_pool.values()]
        
        # Convert each question into the specified format
        failed_count = 0
        with open(OUTPUT_PATH, "w") as fout:
            for entry in tqdm(questions):
                try:
                    query = entry["question"][0][0]["content"]
                    
                    # Keep original tools
                    original_tools = [format_tool(f) for f in entry["function"]]
                    
                    # Select additional tools from pool (excluding ones already in original_tools)
                    additional_tools = []
                    original_tool_names = {t["name"] for t in original_tools}
                    available_tools = [t for t in pool_tools if t["name"] not in original_tool_names]
                    
                    # Select up to 70 additional tools if available
                    num_additional = min(70, len(available_tools))
                    if num_additional > 0:
                        additional_tools = random.sample(available_tools, num_additional)
                    
                    # Combine original and additional tools
                    all_tools = original_tools + additional_tools
                    
                    # Create the new format dictionary
                    new_format_entry = {
                        "format": FORMAT_VALUE,
                        "system_prompt": SYSTEM_PROMPT,
                        "user_query": [query],
                        "plan": "",
                        "tool_defs": all_tools,
                        "skill_trajectory": ["None of the available tools are suitable for this request."],
                        "reason_for_skills": ["The query cannot be fulfilled using the provided tools."],
                        "expected_result": [],
                        "execution": []
                    }
                    
                    fout.write(json.dumps(new_format_entry) + "\n")
                except Exception as e:
                    print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
                    failed_count += 1
                    continue

        print(f"✅ Done writing file for {QUESTION_PATH}. Failed entries: {failed_count}/{len(questions)}")

if __name__ == "__main__":
    main()

Processing /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_multiple.json


100%|██████████| 1053/1053 [00:00<00:00, 4130.59it/s]

✅ Done writing file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_multiple.json. Failed entries: 0/1053


# Ground truth inside the original file

In [163]:
import json
import random
from tqdm import tqdm
import os
import re

# === Helper Functions ===
def flatten_parameters(func):
    """Flatten the nested parameters structure."""
    return func.get("parameters", {}).get("properties", {})

def format_tool(func):
    """Format the function into a tool dictionary for XLAM."""
    return {
        "name": func["name"],
        "description": func.get("description", ""),
        "parameters": flatten_parameters(func)
    }

def parse_ground_truth_string(gt_str):
    """Parse ground truth from function call string format.
    Example: "math_lcm(a=24, b=18)" -> ("math_lcm", {"a": 24, "b": 18})
    """
    try:
        # Extract function name and arguments using regex
        match = re.match(r"(\w+)\((.*)\)", gt_str)
        if not match:
            return None, None
        
        func_name = match.group(1)
        args_str = match.group(2)
        
        # Parse arguments
        args = {}
        if args_str:
            # Split by comma, handling potential spaces
            arg_pairs = [pair.strip() for pair in args_str.split(",")]
            for pair in arg_pairs:
                if "=" not in pair:  # Skip if there's no equals sign
                    continue
                parts = pair.split("=", 1)  # Split on first equals sign only
                if len(parts) != 2:  # Skip if we don't get exactly two parts
                    continue
                    
                key, value = parts
                # Try to convert to int/float if possible
                try:
                    value = int(value)
                except ValueError:
                    try:
                        value = float(value)
                    except ValueError:
                        # Keep as string if not a number, remove quotes if present
                        value = value.strip('"\'')
                args[key.strip()] = value
                
        return func_name, args
    except Exception as e:
        print(f"Error parsing ground truth '{gt_str}': {str(e)}")
        return None, None

# === Main Processing ===
def main():
    # First, collect all tools from all files to create a global pool
    path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"
    global_tool_pool = {}
    
    # List of files to process
    files_to_process = ["BFCL_v3_exec_multiple.json", "BFCL_v3_exec_simple.json"]
    
    # First pass: collect all tools for the global pool
    for file in files_to_process:
        question_path = os.path.join(path, file)
        with open(question_path, "r") as fq:
            for line in fq:
                entry = json.loads(line.strip())
                for f in entry["function"]:
                    global_tool_pool[f["name"]] = f
    
    # Convert global pool to list of formatted tools
    global_tools = [format_tool(f) for f in global_tool_pool.values()]
    
    # Second pass: process each file
    for file in files_to_process:
        QUESTION_PATH = os.path.join(path, file)
        OUTPUT_PATH = os.path.join(path, "paired_tianhang_formatted", f"{file.split('.')[0]}.jsonl")
        
        print(f"Processing {QUESTION_PATH}")
        
        # Load questions
        questions = []
        with open(QUESTION_PATH, "r") as fq:
            for line in fq:
                if line.strip():
                    questions.append(json.loads(line))
        
        # Process each question
        failed_count = 0
        with open(OUTPUT_PATH, "w") as fout:
            for entry in tqdm(questions):
                try:
                    query = entry["question"][0][0]["content"]
                    
                    # Keep original tools
                    original_tools = [format_tool(f) for f in entry["function"]]
                    original_tool_names = {t["name"] for t in original_tools}
                    
                    # Select additional tools from global pool (excluding ones already in original_tools)
                    available_tools = [t for t in global_tools if t["name"] not in original_tool_names]
                    
                    # Select up to 70 additional tools if available
                    num_additional = min(70, len(available_tools))
                    additional_tools = []
                    if num_additional > 0:
                        additional_tools = random.sample(available_tools, num_additional)
                    
                    # Combine original and additional tools
                    all_tools = original_tools + additional_tools
                    
                    # Extract ground truth from the entry itself
                    ground_truth = entry.get("ground_truth", [""])[0]  # Get first string from list
                    tool_name, tool_args = parse_ground_truth_string(ground_truth)
                    
                    if tool_name is None:
                        failed_count += 1
                        continue
                    
                    # Create the new format dictionary
                    new_format_entry = {
                        "format": FORMAT_VALUE,
                        "system_prompt": SYSTEM_PROMPT,
                        "user_query": [query],
                        "plan": "",
                        "tool_defs": all_tools,
                        "skill_trajectory": [{"name": tool_name, "arguments": tool_args}],
                        "reason_for_skills": [],
                        "expected_result": [],
                        "execution": []
                    }
                    
                    fout.write(json.dumps(new_format_entry) + "\n")
                except Exception as e:
                    print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
                    failed_count += 1
                    continue

        print(f"✅ Done writing file for {QUESTION_PATH}. Failed entries: {failed_count}/{len(questions)}")

if __name__ == "__main__":
    main()

Processing /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_exec_multiple.json


100%|██████████| 50/50 [00:00<00:00, 6898.53it/s]


✅ Done writing file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_exec_multiple.json. Failed entries: 0/50
Processing /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_exec_simple.json


100%|██████████| 100/100 [00:00<00:00, 7099.60it/s]

✅ Done writing file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_exec_simple.json. Failed entries: 0/100


In [164]:
import json

file_path1 = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/paired_tianhang_formatted/BFCL_v3_exec_simple.jsonl"
file_path2 = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/paired_tianhang_formatted/BFCL_v3_exec_multiple_augmented.jsonl"

# Collect skills from both files
skills_simple = set()
skills_multiple = set()

# Helper function to create a hashable representation of a skill
def skill_to_hashable(skill):
    """Convert skill dict to hashable tuple format: (name, frozenset of arg items)"""
    name = skill["name"]
    args = frozenset(skill["arguments"].items())
    return (name, args)

# Read skills from first file
with open(file_path1, "r") as file:
    for line in file:
        entry = json.loads(line)
        for skill in entry["skill_trajectory"]:
            skills_simple.add(skill_to_hashable(skill))

# Read skills from second file
with open(file_path2, "r") as file:
    for line in file:
        entry = json.loads(line)
        for skill in entry["skill_trajectory"]:
            skills_multiple.add(skill_to_hashable(skill))

# Calculate overlap statistics
overlapping_skills = skills_simple & skills_multiple
total_simple = len(skills_simple)
total_multiple = len(skills_multiple)
overlap_count = len(overlapping_skills)

print(f"Statistics:")
print(f"Total unique skills in simple file: {total_simple}")
print(f"Total unique skills in multiple file: {total_multiple}")
print(f"Number of overlapping skills: {overlap_count}")
print(f"Percentage overlap with simple: {(overlap_count/total_simple)*100:.2f}%")
print(f"Percentage overlap with multiple: {(overlap_count/total_multiple)*100:.2f}%")

# Print examples of overlapping skills in a readable format
print("\nExample overlapping skills:")
for name, args in list(overlapping_skills)[:5]:  # Show first 5 overlapping skills
    args_dict = dict(args)
    print(f"- {name}({', '.join(f'{k}={v}' for k, v in args_dict.items())})")

# Also show distribution of skill names
print("\nSkill name distribution in simple:")
simple_names = {}
for name, _ in skills_simple:
    simple_names[name] = simple_names.get(name, 0) + 1
for name, count in sorted(simple_names.items(), key=lambda x: x[1], reverse=True):
    print(f"- {name}: {count} times")

print("\nSkill name distribution in multiple:")
multiple_names = {}
for name, _ in skills_multiple:
    multiple_names[name] = multiple_names.get(name, 0) + 1
for name, count in sorted(multiple_names.items(), key=lambda x: x[1], reverse=True):
    print(f"- {name}: {count} times")

Statistics:
Total unique skills in simple file: 96
Total unique skills in multiple file: 50
Number of overlapping skills: 49
Percentage overlap with simple: 51.04%
Percentage overlap with multiple: 98.00%

Example overlapping skills:
- get_distance(pointB=(48.85, pointA=(45.76)
- calculate_mean(numbers=[22)
- estimate_derivative(function=lambda x: 3*x**2 + 2*x + 1, x=5)
- calculate_nutritional_needs(height=170, weight=59, age=80, gender=female, goal=lose, activity_level=4)
- retrieve_holiday_by_year(country=FR, year=2010)

Skill name distribution in simple:
- sort_array: 2 times
- calculate_nutritional_needs: 2 times
- get_movie_director: 2 times
- calculate_cosine_similarity: 2 times
- retrieve_city_based_on_zipcode: 2 times
- get_zipcode_by_ip_address: 2 times
- get_coordinate_by_ip_address: 2 times
- math_gcd: 2 times
- find_term_on_urban_dictionary: 2 times
- get_distance: 2 times
- get_covid_death_by_country: 2 times
- calculate_final_velocity: 2 times
- get_active_covid_case_by_c

# No need to augment these

In [165]:
import os
question_answer_pairs_3 = {}
path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"
smallset = ["BFCL_v3_live_multiple.json"]
# Traverse the directory and match files to the pair format
for file in os.listdir(path):
    if file in smallset and file.endswith(".json"):
        question_path = os.path.join(path, file)
        answer_path = os.path.join(path, "possible_answer", file)
        output_path = os.path.join(path, "paired_tianhang_formatted", f"{file.split('.')[0]}.jsonl")
        
        question_answer_pairs_3[question_path] = {
            "question_path": question_path,
            "answer_path": answer_path,
            "output_path": output_path
        }


In [166]:
import json
from tqdm import tqdm

# === Helper Functions ===
def flatten_parameters(func):
    """Flatten the nested parameters structure."""
    return func.get("parameters", {}).get("properties", {})

def parse_ground_truth(gt_entry):
    """Extract tool name and arguments from the ground_truth entry."""
    try:
        name, args = list(gt_entry.items())[0]
        flat_args = {k: v[0] if isinstance(v, list) and len(v) > 0 else v for k, v in args.items()}
        return name, flat_args
    except (IndexError, KeyError, AttributeError) as e:
        # Return None to indicate a parsing error
        return None, None

def format_tool(func):
    """Format the function into a tool dictionary for XLAM."""
    return {
        "name": func["name"],
        "description": func.get("description", ""),
        "parameters": flatten_parameters(func)
    }

# === Main Processing ===
def main():
    # Iterate over each question-answer pair
    for paths in question_answer_pairs_3.values():
        QUESTION_PATH = paths["question_path"]
        ANSWER_PATH = paths["answer_path"]
        OUTPUT_PATH = paths["output_path"]
        
        # Check if this is a multiple-function file that doesn't need aggregation
        is_multiple_function = "multiple" in QUESTION_PATH.lower()

        # Load questions and answers from JSONL files
        questions = []
        answers = []
        with open(QUESTION_PATH, "r") as fq:
            for line in fq:
                line = line.strip()
                if line:
                    questions.append(json.loads(line))
        with open(ANSWER_PATH, "r") as fa:
            for line in fa:
                line = line.strip()
                if line:
                    answers.append(json.loads(line))

        # Ensure equal number of question and answer entries
        assert len(questions) == len(answers), "Mismatched number of question and answer entries!"

        # For non-multiple files, collect all unique tools (as before)
        tool_pool = {}
        if not is_multiple_function:
            for q in questions:
                for f in q["function"]:
                    tool_pool[f["name"]] = f
            all_tools = [format_tool(f) for f in tool_pool.values()]

        # Convert each question and corresponding answer
        failed_count = 0
        print(f"Processing {QUESTION_PATH}")
        with open(OUTPUT_PATH, "w") as fout:
            for entry, gt in tqdm(zip(questions, answers), total=len(questions)):
                try:
                    query = entry["question"][0][0]["content"]
                    tool_call_name, tool_args = parse_ground_truth(gt["ground_truth"][0])
                    
                    # Skip entries with parsing errors
                    if tool_call_name is None:
                        failed_count += 1
                        continue
                    
                    # For multiple-function files, use the functions specific to this entry
                    if is_multiple_function:
                        all_tools = [format_tool(f) for f in entry["function"]]
                    
                    # Create the new format dictionary
                    new_format_entry = {
                        "format": FORMAT_VALUE,
                        "system_prompt": SYSTEM_PROMPT,
                        "user_query": [query],
                        "plan": "",
                        "tool_defs": all_tools,
                        "skill_trajectory": [{"name": tool_call_name, "arguments": tool_args}],
                        "reason_for_skills": [],
                        "expected_result": [],
                        "execution": []
                    }
                    
                    fout.write(json.dumps(new_format_entry) + "\n")
                except Exception as e:
                    print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
                    failed_count += 1
                    continue

        print(f"✅ Done writing XLAM file for {QUESTION_PATH}. Failed entries: {failed_count}/{len(questions)}")

if __name__ == "__main__":
    main()

Processing /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_multiple.json


100%|██████████| 1053/1053 [00:00<00:00, 45317.07it/s]

✅ Done writing XLAM file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_multiple.json. Failed entries: 0/1053


# No possible answer for these

In [167]:
data = {"BFCL_v3_live_irrelevance.json"}


In [168]:
import json
from tqdm import tqdm

# === Helper Functions ===
def flatten_parameters(func):
    """Flatten the nested parameters structure."""
    return func.get("parameters", {}).get("properties", {})

def format_tool(func):
    """Format the function into a tool dictionary for XLAM."""
    return {
        "name": func["name"],
        "description": func.get("description", ""),
        "parameters": flatten_parameters(func)
    }

# === Main Processing ===
def main():
    # Iterate over each question path
    path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data"
    for file in data:
        QUESTION_PATH = os.path.join(path, file)
        OUTPUT_PATH = os.path.join(path, "paired_tianhang_formatted", f"{file.split('.')[0]}.jsonl")
        
        print(f"Processing {QUESTION_PATH}")
        
        # Load questions from JSONL file
        questions = []
        with open(QUESTION_PATH, "r") as fq:
            for line in fq:
                line = line.strip()
                if line:
                    questions.append(json.loads(line))

        # Convert each question into the specified format
        failed_count = 0
        with open(OUTPUT_PATH, "w") as fout:
            for entry in tqdm(questions):
                try:
                    query = entry["question"][0][0]["content"]
                    
                    # Format tools from the entry
                    all_tools = [format_tool(f) for f in entry["function"]]
                    
                    # Create the new format dictionary with the "no tools" message
                    new_format_entry = {
                        "format": FORMAT_VALUE,
                        "system_prompt": SYSTEM_PROMPT,
                        "user_query": [query],
                        "plan": "",
                        "tool_defs": all_tools,
                        "skill_trajectory": ["None of the available tools are suitable for this request."],
                        "reason_for_skills": ["The query cannot be fulfilled using the provided tools."],
                        "expected_result": [],
                        "execution": []
                    }
                    
                    fout.write(json.dumps(new_format_entry) + "\n")
                except Exception as e:
                    print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
                    failed_count += 1
                    continue

        print(f"✅ Done writing file for {QUESTION_PATH}. Failed entries: {failed_count}/{len(questions)}")

if __name__ == "__main__":
    main()

Processing /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_irrelevance.json


  0%|          | 0/882 [00:00<?, ?it/s]

100%|██████████| 882/882 [00:00<00:00, 45242.35it/s]

✅ Done writing file for /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/BFCL_v3_live_irrelevance.json. Failed entries: 0/882


# Total Stats

In [169]:
import os

# Directory containing the jsonl files
directory_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/paired_tianhang_formatted"

# Initialize a counter for the total number of entries
total_entries = 0

# Iterate through each file in the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".jsonl"):
        file_path = os.path.join(directory_path, filename)
        with open(file_path, "r") as file:
            # Count the number of lines (entries) in the current file
            num_entries = sum(1 for line in file)
            total_entries += num_entries

# Print the total number of entries
print(f"Total number of entries: {total_entries}")


Total number of entries: 102720


In [170]:
import os
import json
from collections import defaultdict
from statistics import mean, median

# Directory containing the jsonl files
directory_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/paired_tianhang_formatted"

# Dictionary to store stats for each file
file_stats = {}

# Iterate through each file in the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".jsonl"):
        file_path = os.path.join(directory_path, filename)
        tool_counts = []
        unique_tools = set()
        entry_count = 0
        
        with open(file_path, "r") as file:
            for line in file:
                entry_count += 1
                data = json.loads(line)
                if 'tool_defs' in data:
                    tools = data['tool_defs']
                    tool_counts.append(len(tools))
                    # Track unique tool names
                    for tool in tools:
                        unique_tools.add(tool['name'])
        
        # Calculate statistics
        file_stats[filename] = {
            'total_entries': entry_count,
            'min_tools': min(tool_counts),
            'max_tools': max(tool_counts),
            'mean_tools': round(mean(tool_counts), 2),
            'median_tools': median(tool_counts),
            'unique_tools': len(unique_tools)
        }

# Print statistics in a readable format
print("Tool Definition Statistics Per File:\n")
for filename, stats in sorted(file_stats.items()):
    print(f"📁 {filename}")
    print(f"   Total entries: {stats['total_entries']}")
    print(f"   Tools per entry:")
    print(f"      Minimum: {stats['min_tools']}")
    print(f"      Maximum: {stats['max_tools']}")
    print(f"      Mean: {stats['mean_tools']}")
    print(f"      Median: {stats['median_tools']}")
    print(f"   Unique tools across all entries: {stats['unique_tools']}")
    print()

# Calculate overall statistics
all_entries = sum(stats['total_entries'] for stats in file_stats.values())
all_unique_tools = set()
for filename, stats in file_stats.items():
    with open(os.path.join(directory_path, filename), "r") as file:
        for line in file:
            data = json.loads(line)
            if 'tool_defs' in data:
                for tool in data['tool_defs']:
                    all_unique_tools.add(tool['name'])

print("📊 Overall Statistics:")
print(f"Total entries across all files: {all_entries}")
print(f"Total unique tools across all files: {len(all_unique_tools)}")

Tool Definition Statistics Per File:

📁 BFCL_v3_exec_multiple.jsonl
   Total entries: 50
   Tools per entry:
      Minimum: 71
      Maximum: 71
      Mean: 71
      Median: 71.0
   Unique tools across all entries: 71

📁 BFCL_v3_exec_multiple_augmented.jsonl
   Total entries: 50
   Tools per entry:
      Minimum: 71
      Maximum: 71
      Mean: 71
      Median: 71.0
   Unique tools across all entries: 71

📁 BFCL_v3_exec_simple.jsonl
   Total entries: 100
   Tools per entry:
      Minimum: 71
      Maximum: 71
      Mean: 71
      Median: 71.0
   Unique tools across all entries: 71

📁 BFCL_v3_java.jsonl
   Total entries: 100
   Tools per entry:
      Minimum: 100
      Maximum: 100
      Mean: 100
      Median: 100.0
   Unique tools across all entries: 100

📁 BFCL_v3_javascript.jsonl
   Total entries: 50
   Tools per entry:
      Minimum: 50
      Maximum: 50
      Mean: 50
      Median: 50.0
   Unique tools across all entries: 50

📁 BFCL_v3_live_irrelevance.jsonl
   Total entries: 882

# External

In [141]:
import json

def format_tool(api_tool):
    """Convert API tool format to our tool definition format."""
    return {
        "name": api_tool["apiCode"],
        "description": api_tool["description"],
        "parameters": api_tool.get("parameters", {}),
        "response": api_tool.get("response", {})
    }

def process_file(input_path, output_path):
    with open(output_path, "w") as fout:
        with open(input_path, "r") as fin:
            for line in fin:  # Changed 'file' to 'fin'
                entry = json.loads(line)
                
                new_format_entry = {
                    "format": FORMAT_VALUE,
                    "system_prompt": SYSTEM_PROMPT,
                    "user_query": [entry["query"]],
                    "plan": "",
                    "tool_defs": [format_tool(tool) for tool in entry["tools"]],
                    "skill_trajectory": entry["answers"],
                    "reason_for_skills": [],
                    "expected_result": [],
                    "execution": []
                }
                
                fout.write(json.dumps(new_format_entry) + "\n")

    print(f"✅ Done writing formatted file: {output_path}")

# Example usage
input_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/apibank_xlamformatted.jsonl"
output_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/apibank_reformatted.jsonl"
process_file(input_path, output_path)

✅ Done writing formatted file: /Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/apibank_reformatted.jsonl


In [142]:
def get_jsonl_length(file_path):
    with open(file_path, "r") as f:
        return sum(1 for _ in f)

output_length = get_jsonl_length(output_path)
print(f"Length of the output_path jsonl: {output_length}")


Length of the output_path jsonl: 33416


In [143]:
import json

def format_tool(tool):
    """Convert tool format to our tool definition format."""
    return {
        "name": tool["name"],
        "description": tool["description"],
        "parameters": tool["parameters"],
        "response": {
            "data": {
                "description": tool.get("output", ""),
                "type": "object"
            }
        }
    }

def parse_arguments(arg_str):
    """Safely parse arguments string to dict."""
    try:
        if isinstance(arg_str, dict):
            return arg_str
        if isinstance(arg_str, str):
            try:
                return json.loads(arg_str)
            except json.JSONDecodeError:
                # If direct parsing fails, try to clean the string
                cleaned_str = arg_str.replace("'", '"').strip()
                return json.loads(cleaned_str)
        return {}
    except Exception as e:
        print(f"Error parsing arguments: {str(e)}")
        print(f"Problematic arguments string: {arg_str}")
        return {}

def process_file(input_path, output_path):
    with open(output_path, "w") as fout:
        with open(input_path, "r") as fin:
            for line_num, line in enumerate(fin, 1):
                try:
                    entry = json.loads(line)
                    
                    # Handle cases where there are no answers
                    answers = entry.get("answers", [])
                    if answers:
                        answer = answers[0]
                        reasoning = answer.get("reasoning", "NaN")
                        execution_output = answer.get("execution_output", "NaN")
                        
                        # Safely parse arguments
                        args = parse_arguments(answer.get("arguments", "{}"))
                        
                        skill_trajectory = [{
                            "name": answer["name"],
                            "arguments": args
                        }]
                    else:
                        reasoning = []
                        execution_output = []
                        skill_trajectory = []
                    
                    new_format_entry = {
                        "format": FORMAT_VALUE,
                        "system_prompt": SYSTEM_PROMPT,
                        "user_query": [entry["query"]],
                        "plan": "",
                        "tool_defs": [format_tool(tool) for tool in entry["tools"]],
                        "skill_trajectory": skill_trajectory,
                        "reason_for_skills": [reasoning],
                        "expected_result": "",
                        "execution": [execution_output]
                    }
                    
                    fout.write(json.dumps(new_format_entry) + "\n")
                except Exception as e:
                    print(f"Error processing line {line_num}: {str(e)}")
                    continue

    print(f"✅ Done writing formatted file: {output_path}")

# Example usage
input_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/processed_data.jsonl"
output_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/processed_data_reformatted.jsonl"
process_file(input_path, output_path)

Error parsing arguments: Extra data: line 1 column 34 (char 33)
Problematic arguments string: {"animalType": "dog", "count": 2}, {"animalType": "cat", "count": 3}, {"animalType": "bird", "count": 1}
Error parsing arguments: Expecting value: line 1 column 19 (char 18)
Problematic arguments string: {"point": {"lat": <user's current latitude>, "lng": <user's current longitude>}, "locale": "en"}
Error parsing arguments: Expecting ',' delimiter: line 1 column 16 (char 15)
Problematic arguments string: {"routeId": 809P-10}
Error parsing arguments: Expecting value: line 1 column 45 (char 44)
Problematic arguments string: {"text": "tourist attractions", "latitude": <city's latitude>, "longitude": <city's longitude>}
Error parsing arguments: Expecting value: line 1 column 81 (char 80)
Problematic arguments string: {"addresses": ["123 Main Street, Anytown, USA", "456 Elm Street, Anytown, USA", ...]}
Error parsing arguments: Expecting value: line 1 column 56 (char 55)
Problematic arguments string

In [144]:
def get_jsonl_length(file_path):
    with open(file_path, "r") as f:
        return sum(1 for _ in f)

output_length = get_jsonl_length(output_path)
print(f"Length of the output_path jsonl: {output_length}")


Length of the output_path jsonl: 4255


In [147]:
def process_file(input_path: str, output_path: str):
    """Process the JSON file and convert to the required format."""
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with open(input_path, 'r') as f:
        data = json.load(f)
    
    formatted_entries = []
    
    for entry in data:
        try:
            # Parse the tools and answers - handle both string and list/dict formats
            try:
                tools = entry['tools'] if isinstance(entry['tools'], list) else json.loads(entry['tools'])
                answers = entry['answers'] if isinstance(entry['answers'], list) else json.loads(entry['answers'])
            except json.JSONDecodeError as e:
                print(f"JSON decode error for entry {entry.get('id', 'unknown')}: {str(e)}")
                print(f"Tools type: {type(entry['tools'])}")
                print(f"Answers type: {type(entry['answers'])}")
                continue
            
            # Format each answer into the skill trajectory
            skill_trajectory = []
            for answer in answers:
                # Handle both string and dict formats for answer
                if isinstance(answer, str):
                    answer = json.loads(answer)
                
                skill = {
                    "name": answer['name'],
                    "args": answer['arguments']
                }
                skill_trajectory.append(skill)
            
            # Create the formatted entry
            formatted_entry = {
                "format": FORMAT_VALUE,
                "system_prompt": SYSTEM_PROMPT,
                "user_query": [entry["query"]],
                "plan": "",
                "tool_defs": [format_tool(tool) for tool in tools],  # Use parsed tools directly
                "skill_trajectory": skill_trajectory,
                "reason_for_skills": [],
                "expected_result": [],
                "available_functions": tools
            }
            
            formatted_entries.append(formatted_entry)
            
        except Exception as e:
            print(f"Error processing entry {entry.get('id', 'unknown')}: {str(e)}")
            print(f"Entry content: {entry}")  # Add this for debugging
            continue
    
    # Write the formatted entries to the output file
    with open(output_path, 'w') as f:
        for entry in formatted_entries:
            f.write(json.dumps(entry) + '\n')
            
def main():
    input_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/xlam_function_calling_60k.json"
    output_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/external/xlam_function_calling_60k_reformatted.jsonl"
    
    process_file(input_path, output_path)

if __name__ == "__main__":
    main()

In [148]:
import os

def count_total_lines_in_directory(directory_path):
    total_lines = 0
    for filename in os.listdir(directory_path):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(directory_path, filename)
            with open(file_path, 'r') as file:
                total_lines += sum(1 for _ in file)
    return total_lines

directory_path = "/Users/ardademirci/Desktop/altera/gorilla/berkeley-function-call-leaderboard/data/paired_tianhang_formatted"
total_lines = count_total_lines_in_directory(directory_path)
print(f"Total number of lines across all JSONL files: {total_lines}")


Total number of lines across all JSONL files: 102720
